# Grid Event Summary

Generate an automated daily operations briefing from grid event logs.

**Prerequisites:** Ollama running with `qwen2.5:3b` pulled. Install: `pip install requests pandas`

**Note:** Synthetic event data. In production, connect to your EMS/SCADA event historian.

In [ ]:
import pandas as pd
import requests

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'qwen2.5:3b'

## Generate Synthetic Grid Events

In [ ]:
events = [
    {'timestamp': '2026-03-10 00:12', 'type': 'ALARM', 'severity': 'LOW', 'equip': 'CAP-102', 'sub': 'Elm St', 'desc': 'Capacitor bank switched off -- low VAR demand'},
    {'timestamp': '2026-03-10 02:45', 'type': 'STATUS', 'severity': 'INFO', 'equip': 'T-201', 'sub': 'Oak Park', 'desc': 'Tap changer operated -- position 7 to 8'},
    {'timestamp': '2026-03-10 05:30', 'type': 'STATUS', 'severity': 'INFO', 'equip': 'GEN-01', 'sub': 'Central', 'desc': 'Peaking unit startup for morning ramp'},
    {'timestamp': '2026-03-10 06:15', 'type': 'ALARM', 'severity': 'MEDIUM', 'equip': 'FDR-305', 'sub': 'Riverside', 'desc': 'Overcurrent relay pickup 1.2x normal, auto-reclosed OK'},
    {'timestamp': '2026-03-10 07:02', 'type': 'ALARM', 'severity': 'HIGH', 'equip': 'T-101', 'sub': 'Main', 'desc': 'Oil temperature 92C -- approaching 95C alarm limit'},
    {'timestamp': '2026-03-10 07:15', 'type': 'STATUS', 'severity': 'INFO', 'equip': 'T-101', 'sub': 'Main', 'desc': 'Cooling fans stage 2 activated'},
    {'timestamp': '2026-03-10 07:45', 'type': 'STATUS', 'severity': 'INFO', 'equip': 'T-101', 'sub': 'Main', 'desc': 'Oil temperature decreasing -- 88C, fans effective'},
    {'timestamp': '2026-03-10 09:30', 'type': 'ALARM', 'severity': 'LOW', 'equip': 'BRK-410', 'sub': 'Ind. Park', 'desc': 'SF6 pressure low -- 68 psi (alarm at 70)'},
    {'timestamp': '2026-03-10 11:22', 'type': 'ALARM', 'severity': 'HIGH', 'equip': 'FDR-207', 'sub': 'Oak Park', 'desc': 'Feeder lockout -- 3 unsuccessful reclose attempts'},
    {'timestamp': '2026-03-10 11:22', 'type': 'EVENT', 'severity': 'HIGH', 'equip': 'FDR-207', 'sub': 'Oak Park', 'desc': 'Crew dispatched -- ~1,200 customers affected'},
    {'timestamp': '2026-03-10 12:45', 'type': 'EVENT', 'severity': 'MEDIUM', 'equip': 'FDR-207', 'sub': 'Oak Park', 'desc': 'Fault located -- tree contact at pole 47'},
    {'timestamp': '2026-03-10 13:30', 'type': 'STATUS', 'severity': 'INFO', 'equip': 'FDR-207', 'sub': 'Oak Park', 'desc': 'Feeder restored. All customers re-energized.'},
    {'timestamp': '2026-03-10 16:55', 'type': 'ALARM', 'severity': 'LOW', 'equip': 'RTU-ELM', 'sub': 'Elm St', 'desc': 'RTU comm timeout 30s. Auto-recovered.'},
    {'timestamp': '2026-03-10 21:10', 'type': 'ALARM', 'severity': 'MEDIUM', 'equip': 'T-101', 'sub': 'Main', 'desc': 'Buchholz relay gas accumulation alarm. No trip. Gas sample scheduled.'},
]
df = pd.DataFrame(events)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Loaded {len(df)} events for March 10, 2026')
print(f"\nBy severity: {df['severity'].value_counts().to_dict()}")
df[['timestamp', 'severity', 'equip', 'desc']]

## Generate Morning Briefing

In [ ]:
event_log = df.to_string(index=False)

prompt = f"""You are the night shift supervisor preparing the morning briefing.

Grid events from March 10, 2026:

{event_log}

Write a concise morning briefing:
1. Executive summary (2-3 sentences)
2. Significant events (HIGH/MEDIUM with resolution status)
3. Equipment requiring follow-up today
4. Open items for day shift"""

response = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': prompt}],
    'stream': False,
})

print('=' * 60)
print('DAILY OPERATIONS BRIEFING -- March 11, 2026 06:00')
print('Prepared by: AI Assistant (FERCoff Sandbox)')
print('=' * 60)
print()
print(response.json()['message']['content'])

## Drill Down: T-101 Correlation Analysis

In [ ]:
t101_events = df[df['equip'] == 'T-101'].to_string(index=False)

followup = f"""T-101 at Main substation had these events yesterday:

{t101_events}

Context: DGA from January showed acetylene at 5 ppm (rising). Emergency DGA March 1 showed 8 ppm (above 7 ppm action level).

As a transformer engineer:
1. Are the thermal event and Buchholz alarm likely related?
2. What does rising acetylene + Buchholz gas suggest?
3. Recommended action and timeline?"""

response = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': followup}],
    'stream': False,
})
print(response.json()['message']['content'])

## Summary

Demonstrated automated morning briefing generation and drill-down correlation analysis. The T-101 scenario (thermal + Buchholz + rising DGA) shows how AI connects dots across data sources typically siloed in different systems.